# 04 — Canonical Gold referral model and snapshots

Build Gold referral facts and KPI views from the current Silver state. When
`AS_OF_DATE` is supplied by the archive replay notebook, calculations and the
snapshot use that historical export date. With a blank parameter, the notebook
uses the current date for the live pipeline.


In [ ]:
AS_OF_DATE = ""  # Optional YYYY-MM-DD; archive replay passes the month-end export date.
GOLD_SCHEMA = "gold"
SNAPSHOT_TABLE = "gold.fact_referral_snapshot"

# Shared configuration setup
CFG_NOTEBOOK_NAME = "00_setup_cfg"
AUDIT_TABLE = "monitoring.cfg_silver_export_load"
TIME_PARSER_POLICY = "CORRECTED"
JOB_RUN_ID = ""  # Parent orchestration correlation ID.
NOTEBOOK_TIMEOUT_SECONDS = 1800


In [ ]:
from notebookutils import mssparkutils

cfg_result = mssparkutils.notebook.run(
    CFG_NOTEBOOK_NAME,
    NOTEBOOK_TIMEOUT_SECONDS,
    {"AUDIT_TABLE": AUDIT_TABLE, "TIME_PARSER_POLICY": TIME_PARSER_POLICY},
)
print(f"Configuration setup completed: {cfg_result}")


In [ ]:
from datetime import date, datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F

if AS_OF_DATE:
    AS_OF_DATE_VALUE = datetime.strptime(AS_OF_DATE, "%Y-%m-%d").date()
else:
    AS_OF_DATE_VALUE = date.today()
AS_OF_SQL = f"DATE '{AS_OF_DATE_VALUE.isoformat()}'"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
print(f"Gold as-of date: {AS_OF_DATE_VALUE}")


In [ ]:
GOLD_SOURCE_REQUIREMENTS = {
    "silver.referral": {
        "referral_id", "required_start_date", "response_required_by_date",
        "placement_type", "referral_created_date", "referral_modified_date",
        "referral_status", "export_date",
    },
    "silver.offer": {
        "offer_id", "referral_provider_id", "offer_status", "provider_home_id",
        "offer_date", "last_modified_date",
    },
    "silver.referral_provider": {
        "referral_provider_id", "referral_id", "provider_id",
    },
    "silver.ipa": {
        "referral_id", "created_datetime", "updated_datetime",
        "placement_admission_date", "costs_total_weekly_fee",
    },
    "silver.referral_lifecycle_event": {
        "event_id", "referral_id", "event_type", "event_timestamp",
        "sequence_number", "created_by", "created_timestamp",
    },
}

gold_input_issues = []
for table_name, required_columns in GOLD_SOURCE_REQUIREMENTS.items():
    if not spark.catalog.tableExists(table_name):
        gold_input_issues.append(f"{table_name}: table is missing")
        continue
    actual_columns = {field.name.lower() for field in spark.table(table_name).schema.fields}
    missing_columns = sorted(required_columns - actual_columns)
    if missing_columns:
        gold_input_issues.append(f"{table_name}: missing {missing_columns}")

if gold_input_issues:
    raise RuntimeError(
        "Gold source validation failed. "
        + "; ".join(gold_input_issues)
        + ". Deploy the current setup notebook so monitoring.cfg_schema_contract_column is refreshed, then rerun "
          "Silver for this snapshot before 04_gold_model."
    )

print("Gold source validation passed")
# Lifecycle events are an explicit Silver derivation from referral, offer, and
# IPA timestamps; they are not a source-system referral-event audit log.
EVENT_ROLLUP_SOURCE = "silver.referral_lifecycle_event"



In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW gold.fact_referral AS
WITH referral_history AS (
  SELECT *, ROW_NUMBER() OVER (
    PARTITION BY referral_id
    ORDER BY COALESCE(referral_modified_date, referral_created_date, export_date) DESC,
             export_date DESC
  ) AS row_number_current
  FROM silver.referral
),
referral_current AS (
  SELECT * FROM referral_history WHERE row_number_current = 1
),
referral_created AS (
  SELECT referral_id, MIN(referral_created_date) AS ReferralCreatedDate
  FROM silver.referral GROUP BY referral_id
),
offer_rollup AS (
  SELECT rp.referral_id,
    MIN(o.offer_date) AS FirstOfferDate,
    MIN(CASE WHEN LOWER(o.offer_status) IN ('accepted','approved','selected')
        THEN o.last_modified_date END) AS OfferAcceptedDate,
    COUNT(DISTINCT o.offer_id) AS OfferCount,
    COUNT(DISTINCT o.provider_home_id) AS UniqueHomesOffered,
    MAX(COALESCE(o.last_modified_date, o.offer_date)) AS LastOfferActivityDate
  FROM silver.offer o
  INNER JOIN silver.referral_provider rp
    ON o.referral_provider_id = rp.referral_provider_id
  GROUP BY rp.referral_id
),
ipa_rollup AS (
  SELECT referral_id, MIN(created_datetime) AS IPAIssuedDate,
    MIN(placement_admission_date) AS PlannedPlacementStartDate,
    SUM(costs_total_weekly_fee) AS EstimatedWeeklyCost,
    MAX(COALESCE(updated_datetime, created_datetime)) AS LastIPAActivityDate
  FROM silver.ipa GROUP BY referral_id
),
event_rollup AS (
  SELECT referral_id, MIN(event_timestamp) AS FirstActionDate,
    MAX(COALESCE(event_timestamp, created_timestamp)) AS LastEventActivityDate
  FROM {EVENT_ROLLUP_SOURCE} GROUP BY referral_id
),
base AS (
  SELECT r.referral_id AS ReferralID, c.ReferralCreatedDate,
    r.required_start_date AS RequiredPlacementDate,
    r.response_required_by_date AS ResponseRequiredDate,
    r.referral_modified_date AS ReferralModifiedTimestamp,
    r.referral_status AS CurrentStatus, r.placement_type AS PlacementTypeRequired,
    e.FirstActionDate, o.FirstOfferDate, o.OfferAcceptedDate, i.IPAIssuedDate,
    CASE WHEN LOWER(COALESCE(r.referral_status, '')) IN ('closed','cancelled','withdrawn','completed')
      THEN COALESCE(r.referral_modified_date, r.export_date) END AS ReferralClosedDate,
    CAST(NULL AS STRING) AS ReferralClosureReason,
    GREATEST(COALESCE(r.referral_modified_date, r.referral_created_date, r.export_date),
      e.LastEventActivityDate, o.LastOfferActivityDate, i.LastIPAActivityDate) AS LastActivityDate,
    o.OfferCount, o.UniqueHomesOffered,
    i.PlannedPlacementStartDate, i.EstimatedWeeklyCost,
    CASE
      WHEN r.required_start_date IS NULL THEN 'Unspecified'
      WHEN DATEDIFF(r.required_start_date, TO_DATE(c.ReferralCreatedDate)) <= 1 THEN 'Critical'
      WHEN DATEDIFF(r.required_start_date, TO_DATE(c.ReferralCreatedDate)) <= 3 THEN 'High'
      WHEN DATEDIFF(r.required_start_date, TO_DATE(c.ReferralCreatedDate)) <= 7 THEN 'Medium'
      ELSE 'Planned'
    END AS PlacementUrgencyBand
  FROM referral_current r
  INNER JOIN referral_created c ON r.referral_id = c.referral_id
  LEFT JOIN offer_rollup o ON r.referral_id = o.referral_id
  LEFT JOIN ipa_rollup i ON r.referral_id = i.referral_id
  LEFT JOIN event_rollup e ON r.referral_id = e.referral_id
)
SELECT {AS_OF_SQL} AS AsOfDate,
  ReferralID, ReferralCreatedDate, RequiredPlacementDate, ResponseRequiredDate,
  FirstActionDate, FirstOfferDate, OfferAcceptedDate, IPAIssuedDate,
  ReferralClosedDate, ReferralClosureReason, LastActivityDate, CurrentStatus,
  PlacementTypeRequired, PlacementUrgencyBand,
  CAST(NULL AS STRING) AS ChildCriticalityCode,
  COALESCE(OfferCount, 0) AS OfferCount,
  COALESCE(UniqueHomesOffered, 0) AS UniqueHomesOffered,
  COALESCE(OfferCount, 0) > 0 AS HasOffer,
  DATEDIFF(TO_DATE(FirstActionDate), TO_DATE(ReferralCreatedDate)) AS DaysToFirstAction,
  DATEDIFF(TO_DATE(FirstOfferDate), TO_DATE(ReferralCreatedDate)) AS DaysToFirstOffer,
  DATEDIFF(TO_DATE(OfferAcceptedDate), TO_DATE(ReferralCreatedDate)) AS DaysToAcceptedOffer,
  DATEDIFF(TO_DATE(IPAIssuedDate), TO_DATE(ReferralCreatedDate)) AS DaysToIPA,
  DATEDIFF(COALESCE(TO_DATE(ReferralClosedDate), {AS_OF_SQL}),
    TO_DATE(ReferralCreatedDate)) AS DaysOpen,
  DATEDIFF({AS_OF_SQL}, TO_DATE(LastActivityDate)) AS DaysWithoutActivity,
  CASE WHEN RequiredPlacementDate IS NOT NULL AND RequiredPlacementDate < {AS_OF_SQL}
    THEN DATEDIFF({AS_OF_SQL}, RequiredPlacementDate) ELSE 0 END AS DaysPastRequiredDate,
  LOWER(COALESCE(CurrentStatus, '')) NOT IN
    ('closed','cancelled','withdrawn','completed') AS IsOpen,
  IPAIssuedDate IS NOT NULL AND RequiredPlacementDate IS NOT NULL
    AND TO_DATE(IPAIssuedDate) <= RequiredPlacementDate AS PlacedByRequiredDate,
  CASE
    WHEN IPAIssuedDate IS NOT NULL AND RequiredPlacementDate IS NOT NULL
      AND TO_DATE(IPAIssuedDate) <= RequiredPlacementDate THEN 'Placed by target'
    WHEN IPAIssuedDate IS NOT NULL THEN 'Placed after target'
    WHEN RequiredPlacementDate < {AS_OF_SQL} AND LOWER(COALESCE(CurrentStatus, '')) NOT IN
      ('closed','cancelled','withdrawn','completed') THEN 'Open overdue'
    WHEN LOWER(COALESCE(CurrentStatus, '')) NOT IN
      ('closed','cancelled','withdrawn','completed') THEN 'Open on track'
    ELSE 'Closed without placement'
  END AS RequiredPlacementDateOutcome,
  PlannedPlacementStartDate, EstimatedWeeklyCost,
  CURRENT_TIMESTAMP() AS GoldModelledAt
FROM base
WHERE TO_DATE(ReferralCreatedDate) <= {AS_OF_SQL}
""")


In [ ]:
snapshot = spark.table("gold.fact_referral").select(
    F.lit(AS_OF_DATE_VALUE).cast("date").alias("SnapshotDate"),
    "ReferralID", "CurrentStatus", "PlacementUrgencyBand", "RequiredPlacementDate",
    "IsOpen", "HasOffer", "OfferCount", "DaysOpen", "DaysWithoutActivity",
    "DaysPastRequiredDate", "PlacedByRequiredDate", "RequiredPlacementDateOutcome",
)
if not spark.catalog.tableExists(SNAPSHOT_TABLE):
    snapshot.write.format("delta").mode("overwrite").saveAsTable(SNAPSHOT_TABLE)
else:
    target = DeltaTable.forName(spark, SNAPSHOT_TABLE)
    (target.alias("t").merge(snapshot.alias("s"),
        "t.SnapshotDate = s.SnapshotDate AND t.ReferralID = s.ReferralID")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
print(f"Snapshot refreshed for {AS_OF_DATE_VALUE}: {snapshot.count():,} referrals")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW gold.fact_referral_lifecycle_event AS
SELECT event_id AS EventID, referral_id AS ReferralID, event_type AS EventType,
  event_timestamp AS EventTimestamp, sequence_number AS SequenceNumber,
  created_by AS CreatedBy
FROM {EVENT_ROLLUP_SOURCE}
""")
spark.sql("""
CREATE OR REPLACE VIEW gold.vw_kpi_referral_board_summary AS
SELECT AsOfDate, PlacementUrgencyBand, RequiredPlacementDateOutcome,
  COUNT(DISTINCT ReferralID) AS ReferralCount,
  SUM(CASE WHEN IsOpen THEN 1 ELSE 0 END) AS OpenReferralCount,
  SUM(CASE WHEN IsOpen AND RequiredPlacementDate < AsOfDate THEN 1 ELSE 0 END) AS OpenOverdueCount,
  SUM(CASE WHEN PlacedByRequiredDate THEN 1 ELSE 0 END) AS PlacedByRequiredDateCount,
  SUM(CASE WHEN HasOffer THEN 1 ELSE 0 END) AS ReferralsWithOfferCount,
  PERCENTILE_APPROX(DaysToIPA, 0.5) AS MedianDaysToIPA,
  SUM(COALESCE(EstimatedWeeklyCost, 0)) AS EstimatedWeeklyCost
FROM gold.fact_referral
GROUP BY AsOfDate, PlacementUrgencyBand, RequiredPlacementDateOutcome
""")
spark.sql("""
CREATE OR REPLACE VIEW gold.vw_kpi_referral_monthly AS
SELECT DATE_TRUNC('month', ReferralCreatedDate) AS ReferralCreatedMonth,
  COUNT(DISTINCT ReferralID) AS NewReferralCount,
  SUM(CASE WHEN HasOffer THEN 1 ELSE 0 END) AS ReferralsWithOfferCount,
  SUM(CASE WHEN IPAIssuedDate IS NOT NULL THEN 1 ELSE 0 END) AS IPACount,
  SUM(CASE WHEN PlacedByRequiredDate THEN 1 ELSE 0 END) AS PlacedByRequiredDateCount,
  SUM(CASE WHEN IsOpen THEN 1 ELSE 0 END) AS OpenReferralCount
FROM gold.fact_referral
GROUP BY DATE_TRUNC('month', ReferralCreatedDate)
""")
spark.sql("""
CREATE OR REPLACE VIEW gold.vw_provider_offer_performance AS
SELECT rp.provider_id AS ProviderID,
  COUNT(DISTINCT rp.referral_id) AS ReferralsReceived,
  COUNT(DISTINCT o.offer_id) AS OffersSubmitted,
  COUNT(DISTINCT CASE WHEN LOWER(o.offer_status) IN ('accepted','approved','selected')
    THEN o.offer_id END) AS OffersAccepted,
  COUNT(DISTINCT CASE WHEN f.PlacedByRequiredDate THEN f.ReferralID END) AS ReferralsPlacedByTarget
FROM silver.referral_provider rp
LEFT JOIN silver.offer o ON rp.referral_provider_id = o.referral_provider_id
LEFT JOIN gold.fact_referral f ON rp.referral_id = f.ReferralID
GROUP BY rp.provider_id
""")
